## **XGBoost + lightgbm**

In [ ]:
import zipfile
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import RobustScaler
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import rankdata
import warnings
warnings.filterwarnings("ignore")

# ============================================
# Features loaded from feature_pipeline in cell 1:
#   X_features (880 train users), y_labels
#   X_test (220 unseen test users), test_labels
# ============================================

# ============================================
# HOLD OUT 20% OF TRAINING FOR INTERNAL EVAL
# ============================================
X_trainval, X_heldout, y_trainval, y_heldout = train_test_split(
    X_features, y_labels, test_size=0.2, stratify=y_labels, random_state=42
)

# ============================================
# FEATURE SCALING
# ============================================
scaler = RobustScaler()
X_trainval_scaled = scaler.fit_transform(X_trainval)
X_heldout_scaled = scaler.transform(X_heldout)

# ============================================
# HELPER: build models
# ============================================
def make_xgb(scale_pos_weight, seed=42):
    return xgb.XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.7,
        colsample_bylevel=0.7,
        min_child_weight=3,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1,
    )


def make_lgb(scale_pos_weight, seed=42):
    return lgb.LGBMClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )


# ============================================
# CROSS VALIDATION — XGB + LGB + Ensemble
# ============================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs_xgb, aucs_lgb, aucs_ens = [], [], []
oof_xgb = np.zeros(len(X_trainval))
oof_lgb = np.zeros(len(X_trainval))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval_scaled, y_trainval)):
    X_tr, X_val = X_trainval_scaled[train_idx], X_trainval_scaled[val_idx]
    y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

    spw = np.sum(y_tr == 0) / np.sum(y_tr == 1)

    # --- XGBoost ---
    xgb_model = make_xgb(spw)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    p_xgb = xgb_model.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = p_xgb

    # --- LightGBM ---
    lgb_model = make_lgb(spw)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    p_lgb = lgb_model.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = p_lgb

    # --- Rank-average ensemble ---
    p_ens = (rankdata(p_xgb) + rankdata(p_lgb)) / 2

    auc_xgb = roc_auc_score(y_val, p_xgb)
    auc_lgb = roc_auc_score(y_val, p_lgb)
    auc_ens = roc_auc_score(y_val, p_ens)

    aucs_xgb.append(auc_xgb)
    aucs_lgb.append(auc_lgb)
    aucs_ens.append(auc_ens)

    print(f"Fold {fold+1}:  XGB={auc_xgb:.4f}  LGB={auc_lgb:.4f}  Ensemble={auc_ens:.4f}")

# OOF ensemble AUC
oof_ens = (rankdata(oof_xgb) + rankdata(oof_lgb)) / 2
oof_auc = roc_auc_score(y_trainval, oof_ens)

print("\n=== CV Results ===")
print(f"XGBoost  AUC: {np.mean(aucs_xgb):.4f} ± {np.std(aucs_xgb):.4f}")
print(f"LightGBM AUC: {np.mean(aucs_lgb):.4f} ± {np.std(aucs_lgb):.4f}")
print(f"Ensemble AUC: {np.mean(aucs_ens):.4f} ± {np.std(aucs_ens):.4f}")
print(f"OOF Ensemble: {oof_auc:.4f}")

# ============================================
# FINAL MODELS — trained on all 880 training users
# ============================================
spw = np.sum(y_labels == 0) / np.sum(y_labels == 1)

# Refit scaler on all training data for final model
scaler_final = RobustScaler()
X_all_scaled = scaler_final.fit_transform(X_features)

# Use 10% as early-stopping signal
X_ft, X_es, y_ft, y_es = train_test_split(
    X_all_scaled, y_labels, test_size=0.1, stratify=y_labels, random_state=42
)

final_xgb = make_xgb(spw)
final_xgb.fit(X_ft, y_ft, eval_set=[(X_es, y_es)], verbose=False)

final_lgb = make_lgb(spw)
final_lgb.fit(
    X_ft, y_ft,
    eval_set=[(X_es, y_es)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

# ============================================
# EVALUATE ON HELD-OUT SET (internal, 20% of training)
# ============================================
X_heldout_final = scaler_final.transform(X_heldout)
p_xgb_h = final_xgb.predict_proba(X_heldout_final)[:, 1]
p_lgb_h = final_lgb.predict_proba(X_heldout_final)[:, 1]
y_score_h = (rankdata(p_xgb_h) + rankdata(p_lgb_h)) / 2

print(f"\nXGB + LGB Ensemble (internal held-out):")
print(f"AUC: {roc_auc_score(y_heldout, y_score_h):.4f}")

# ============================================
# EVALUATE ON TEST SET (220 unseen users, excluded from training)
# ============================================
X_test_scaled = scaler_final.transform(X_test)
p_xgb_test = final_xgb.predict_proba(X_test_scaled)[:, 1]
p_lgb_test = final_lgb.predict_proba(X_test_scaled)[:, 1]
y_score_test = (rankdata(p_xgb_test) + rankdata(p_lgb_test)) / 2

# Threshold search on OOF scores
oof_ens_norm = (oof_ens - oof_ens.min()) / (oof_ens.max() - oof_ens.min() + 1e-9)
thresholds = np.linspace(0.01, 0.99, 500)
best_f1, best_threshold = 0, 0.5

for t in thresholds:
    preds_t = (oof_ens_norm >= t).astype(int)
    tp = np.sum((preds_t == 1) & (y_trainval == 1))
    fp = np.sum((preds_t == 1) & (y_trainval == 0))
    fn = np.sum((preds_t == 0) & (y_trainval == 1))
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1_t = 2 * prec * rec / (prec + rec + 1e-9)
    if f1_t > best_f1:
        best_f1, best_threshold = f1_t, t

y_score_test_norm = (y_score_test - y_score_test.min()) / (y_score_test.max() - y_score_test.min() + 1e-9)
y_pred_test = (y_score_test_norm >= best_threshold).astype(int)

print(f"\nXGB + LGB Ensemble (local test set — 220 unseen users):")
print(f"AUC:       {roc_auc_score(test_labels, y_score_test):.4f}")
print(f"Precision: {precision_score(test_labels, y_pred_test, zero_division=0):.4f}")
print(f"Recall:    {recall_score(test_labels, y_pred_test, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(test_labels, y_pred_test, zero_division=0):.4f}")

# ============================================
# SAVE SUBMISSION
# ============================================
y_submit = (y_score_test - y_score_test.min()) / (y_score_test.max() - y_score_test.min())
np.savez("submission.npz", predictions=y_submit)
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.npz", arcname="submission.npz")
print("\nSaved submission.zip — upload to Codabench")

Fold 1:  XGB=0.8398  LGB=0.8672  Ensemble=0.8654
Fold 2:  XGB=0.8612  LGB=0.8486  Ensemble=0.8609
Fold 3:  XGB=0.8759  LGB=0.9123  Ensemble=0.8906
Fold 4:  XGB=0.9491  LGB=0.9319  Ensemble=0.9471
Fold 5:  XGB=0.9812  LGB=0.9776  Ensemble=0.9803

=== CV Results ===
XGBoost  AUC: 0.9014 ± 0.0542
LightGBM AUC: 0.9075 ± 0.0461
Ensemble AUC: 0.9089 ± 0.0471
OOF Ensemble: 0.8950

XGB + LGB Ensemble (internal held-out):
AUC: 0.9693

XGB + LGB Ensemble (local test set — 220 unseen users):
AUC:       0.9479
Precision: 0.7647
Recall:    0.7222
F1 Score:  0.7429

Saved submission.zip — upload to Codabench
